# Faster R-CNN MobileNetV3-Large FPN — Trash Dataset
## 90 epochs | Best theo mAP@50-95 | Val mỗi epoch | Auto Resume

- Dataset giữ nguyên notebook YOLO: `/content/trash_project/Trash/Trash_dataset_balanced`
- 7 classes
- Train 90 epochs
- Sau mỗi epoch → VAL
- BEST = mAP@50-95 VAL cao nhất
- Không dùng TEST
- Model: Faster R-CNN MobileNetV3-Large FPN
- Pretrained COCO
- Input 640×640
- T4: tự thử physical batch 16 → 8 → 4 → 2
- Effective batch = 16 bằng gradient accumulation
- AMP
- Auto resume từ Google Drive

In [ ]:
!pip -q install torchmetrics pycocotools pandas tqdm

import os, glob, math, time, json, random, zipfile
from pathlib import Path
import torch, torchvision
import pandas as pd
import numpy as np

# Giảm RAM/CPU contention trên Colab
torch.set_num_threads(2)
try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
print("Drive:", DRIVE_ROOT)

In [ ]:
zip_file = '/content/drive/MyDrive/Trash (3).zip'
if not os.path.exists(zip_file):
    zip_file = '/content/drive/MyDrive/trash (3).zip'
if zip_file and os.path.exists(zip_file):
    print(f'Extracting: {zip_file} to /content/trash_project...')
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall('/content/trash_project')
    print('Extraction complete.')
else:
    print('Error: Zip file not found on Google Drive.')
    print('Available zip files:', glob.glob('/content/drive/MyDrive/*.zip'))

In [ ]:
CLASS_NAMES = ['glass','battery','cardboard','organic','metal','paper','plastic']
NUM_CLASSES = len(CLASS_NAMES)

dataset_root = Path('/content/trash_project/Trash/Trash_dataset_balanced')
if not dataset_root.exists():
    for root, dirs, files in os.walk('/content/trash_project'):
        if os.path.basename(root) == 'Trash_dataset_balanced':
            dataset_root = Path(root); break
print('Dataset:', dataset_root)
if not dataset_root.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset: {dataset_root}')
SPLITS={'train':dataset_root/'train','val':dataset_root/'val'}
print('Train:', SPLITS['train']); print('Val:', SPLITS['val']); print('Classes:', CLASS_NAMES)

In [ ]:
from PIL import Image
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
REPAIRED_ROOT=Path('/content/frcnn_mobilenet_repaired_labels'); REPAIRED_ROOT.mkdir(parents=True, exist_ok=True)
stats={'train_files':0,'val_files':0,'bad_rows':0,'invalid_class_ids':0,'clipped_values':0,'skipped_zero_area':0,'boxes_written':0}
def find_image(img_root, rel_label):
    for ext in IMG_EXTS:
        p=img_root/rel_label.parent/f'{rel_label.stem}{ext}'
        if p.exists(): return p
    return None
for split in ['train','val']:
    img_root=SPLITS[split]/'images'; label_root=SPLITS[split]/'labels'; out_root=REPAIRED_ROOT/split; out_root.mkdir(parents=True, exist_ok=True)
    files=list(label_root.rglob('*.txt')); stats[f'{split}_files']=len(files)
    for label_path in files:
        rel=label_path.relative_to(label_root); img_path=find_image(img_root,rel)
        if img_path is None: continue
        with Image.open(img_path) as im: W,H=im.size
        out_lines=[]
        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts=line.split()
            if len(parts)!=5: stats['bad_rows']+=1; continue
            try: cls=int(parts[0]); xc,yc,bw,bh=map(float,parts[1:])
            except Exception: stats['bad_rows']+=1; continue
            if not (0<=cls<NUM_CLASSES): stats['invalid_class_ids']+=1; continue
            old=(xc,yc,bw,bh); xc=min(max(xc,0.0),1.0); yc=min(max(yc,0.0),1.0); bw=min(max(bw,0.0),1.0); bh=min(max(bh,0.0),1.0)
            if (xc,yc,bw,bh)!=old: stats['clipped_values']+=1
            xmin=max(0.0,(xc-bw/2)*W); ymin=max(0.0,(yc-bh/2)*H); xmax=min(float(W),(xc+bw/2)*W); ymax=min(float(H),(yc+bh/2)*H)
            if xmax<=xmin or ymax<=ymin: stats['skipped_zero_area']+=1; continue
            out_lines.append(f'{cls} {xmin:.4f} {ymin:.4f} {xmax:.4f} {ymax:.4f}'); stats['boxes_written']+=1
        out_path=out_root/rel; out_path.parent.mkdir(parents=True, exist_ok=True); out_path.write_text('\n'.join(out_lines),encoding='utf-8')
print('=== LABEL REPAIR SUMMARY ===')
for k,v in stats.items(): print(f'{k:24s}: {v:,}')

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
class TrashFasterRCNNDataset(Dataset):
    def __init__(self,split,train=False):
        self.train=train; self.image_dir=SPLITS[split]/'images'; self.label_dir=REPAIRED_ROOT/split; self.samples=[]
        for img_path in sorted(self.image_dir.rglob('*')):
            if img_path.suffix.lower() not in IMG_EXTS: continue
            rel=img_path.relative_to(self.image_dir); self.samples.append((img_path,self.label_dir/rel.with_suffix('.txt')))
    def __len__(self): return len(self.samples)
    def __getitem__(self,idx):
        img_path,label_path=self.samples[idx]; image=Image.open(img_path).convert('RGB'); W,H=image.size; boxes=[]; labels=[]
        if label_path.exists():
            for line in label_path.read_text(encoding='utf-8').splitlines():
                p=line.split()
                if len(p)!=5: continue
                cls=int(p[0]); xmin,ymin,xmax,ymax=map(float,p[1:])
                if xmax<=xmin or ymax<=ymin: continue
                boxes.append([xmin,ymin,xmax,ymax]); labels.append(cls+1)
        if self.train and torch.rand(1).item()<0.5:
            image=TF.hflip(image); boxes=[[W-x2,y1,W-x1,y2] for x1,y1,x2,y2 in boxes]
        image=TF.to_tensor(image); boxes=torch.as_tensor(boxes,dtype=torch.float32).reshape(-1,4); labels=torch.as_tensor(labels,dtype=torch.int64)
        area=((boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])) if len(boxes) else torch.zeros((0,),dtype=torch.float32)
        target={'boxes':boxes,'labels':labels,'image_id':torch.tensor([idx],dtype=torch.int64),'area':area,'iscrowd':torch.zeros((len(labels),),dtype=torch.int64)}
        return image,target
def collate_fn(batch): return tuple(zip(*batch))
train_ds=TrashFasterRCNNDataset('train',True); val_ds=TrashFasterRCNNDataset('val',False)
print('Train images:',len(train_ds)); print('Val images:',len(val_ds))

In [ ]:
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def build_model():
    weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
    model=fasterrcnn_mobilenet_v3_large_fpn(weights=weights,min_size=640,max_size=640)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,NUM_CLASSES+1)
    return model.to(DEVICE)
model=build_model(); total_params=sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,} ({total_params/1e6:.2f}M)'); print('Device:',DEVICE)

In [ ]:
AMP_ENABLED = torch.cuda.is_available()

# =============================================
# T4 BALANCED DataLoader
# 2 workers để tăng tốc CPU -> GPU nhưng vẫn
# hạn chế RAM hệ thống. Không persistent workers.
# =============================================
NUM_WORKERS = 2
PREFETCH_FACTOR = 1

# Giảm thread CPU dư thừa trong mỗi process.
torch.set_num_threads(2)


def make_loader(ds, batch_size, shuffle):
    kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': NUM_WORKERS,
        'collate_fn': collate_fn,
        'pin_memory': True
    }

    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
        kwargs['persistent_workers'] = False

    return DataLoader(ds, **kwargs)


def probe_batch(batch_size):
    probe_model = build_model()
    loader = make_loader(train_ds, batch_size, True)

    images, targets = next(iter(loader))
    images = [x.to(DEVICE, non_blocking=True) for x in images]
    targets = [
        {k: v.to(DEVICE, non_blocking=True) for k, v in t.items()}
        for t in targets
    ]

    ok = False
    try:
        opt = torch.optim.SGD(probe_model.parameters(), lr=1e-4)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=AMP_ENABLED):
            loss_dict = probe_model(images, targets)
            loss = sum(loss_dict.values())
        loss.backward()
        ok = True
    except torch.cuda.OutOfMemoryError:
        ok = False
    finally:
        del probe_model, loader, images, targets
        torch.cuda.empty_cache()

    return ok

# T4 của bạn đã xác nhận batch 32 chạy được.
BATCH_SIZE = None

for candidate in [32, 16, 8, 4, 2]:
    print(f'Testing physical batch = {candidate}')
    if probe_batch(candidate):
        BATCH_SIZE = candidate
        print(' -> OK')
        break
    print(' -> OOM')

if BATCH_SIZE is None:
    raise RuntimeError('GPU không chạy được batch 2.')

GRAD_ACCUM_STEPS = 1

print('Physical batch:', BATCH_SIZE)
print('Accumulation  :', GRAD_ACCUM_STEPS)
print('Effective batch:', BATCH_SIZE * GRAD_ACCUM_STEPS)
print('DataLoader workers:', NUM_WORKERS)
print('Prefetch factor:', PREFETCH_FACTOR)

In [ ]:
train_loader = make_loader(train_ds, BATCH_SIZE, True)
val_loader = make_loader(val_ds, BATCH_SIZE, False)

EPOCHS = 90
LR = 1e-4
WEIGHT_DECAY = 5e-4
GRAD_CLIP_NORM = 5.0

DRIVE_SAVE_DIR = DRIVE_ROOT / 'Trash_FasterRCNN_MobileNetV3_Balanced'
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = DRIVE_SAVE_DIR / 'fasterrcnn_mobilenet_best_map5095.pth'
LAST_CKPT = DRIVE_SAVE_DIR / 'fasterrcnn_mobilenet_last.pth'
HISTORY_CSV = DRIVE_SAVE_DIR / 'fasterrcnn_mobilenet_history.csv'
RESULT_CSV = DRIVE_SAVE_DIR / 'fasterrcnn_mobilenet_val_results.csv'

model = build_model()

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler(
    'cuda',
    enabled=AMP_ENABLED
)

print('BEST:', BEST_CKPT)
print('LAST:', LAST_CKPT)
print('Physical batch:', BATCH_SIZE)
print('Effective batch:', BATCH_SIZE * GRAD_ACCUM_STEPS)
print('Workers:', NUM_WORKERS)


In [ ]:
from tqdm.auto import tqdm
from torchmetrics.detection import MeanAveragePrecision
def train_one_epoch(model,loader,optimizer,scaler):
    model.train(); optimizer.zero_grad(set_to_none=True); total_loss=0.0
    for step,(images,targets) in enumerate(tqdm(loader,desc='TRAIN',leave=False)):
        images=[x.to(DEVICE,non_blocking=True) for x in images]; targets=[{k:v.to(DEVICE,non_blocking=True) for k,v in t.items()} for t in targets]
        with torch.amp.autocast(device_type='cuda',enabled=AMP_ENABLED):
            raw_loss=sum(model(images,targets).values()); loss=raw_loss/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step+1)%GRAD_ACCUM_STEPS==0 or (step+1)==len(loader):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP_NORM); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        total_loss+=float(raw_loss.detach().cpu()); tqdm.write if False else None
    return total_loss/max(len(loader),1)
@torch.no_grad()
def validate_map(model,loader):
    model.eval(); metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox')
    for images,targets in tqdm(loader,desc='VAL mAP',leave=False):
        imgs=[x.to(DEVICE,non_blocking=True) for x in images]
        with torch.amp.autocast(device_type='cuda',enabled=AMP_ENABLED): outputs=model(imgs)
        preds=[{'boxes':o['boxes'].detach().cpu(),'scores':o['scores'].detach().cpu(),'labels':o['labels'].detach().cpu()} for o in outputs]
        gts=[{'boxes':t['boxes'].cpu(),'labels':t['labels'].cpu()} for t in targets]
        metric.update(preds,gts)
    r=metric.compute(); return {'mAP50':float(r['map_50']),'mAP50-95':float(r['map']),'mAR100':float(r['mar_100'])}

In [ ]:
if LAST_CKPT.exists():
    print('Found LAST checkpoint. Resuming...'); ckpt=torch.load(LAST_CKPT,map_location=DEVICE,weights_only=False); model.load_state_dict(ckpt['model_state']); optimizer.load_state_dict(ckpt['optimizer_state']); scheduler.load_state_dict(ckpt['scheduler_state']); scaler.load_state_dict(ckpt.get('scaler_state',{})) if 'scaler_state' in ckpt else None; start_epoch=int(ckpt['epoch'])+1; best_map5095=float(ckpt.get('best_map5095',-1.0)); history=ckpt.get('history',[]); print('Resumed after epoch:',ckpt['epoch']); print('Next epoch:',start_epoch); print('Best mAP50-95:',best_map5095)
else:
    print('No checkpoint found. Starting from epoch 1.'); start_epoch=1; best_map5095=-1.0; history=[]
if start_epoch<=EPOCHS:
    start_all=time.time()
    for epoch in range(start_epoch,EPOCHS+1):
        epoch_start=time.time(); train_loss=train_one_epoch(model,train_loader,optimizer,scaler); metrics=validate_map(model,val_loader); map50=metrics['mAP50']; map5095=metrics['mAP50-95']; scheduler.step(); epoch_minutes=(time.time()-epoch_start)/60.0
        row={'epoch':epoch,'train_loss':train_loss,'mAP50':map50,'mAP50-95':map5095,'mAR100':metrics['mAR100'],'lr':optimizer.param_groups[0]['lr'],'epoch_minutes':epoch_minutes}; history=[h for h in history if h['epoch']!=epoch]; history.append(row); is_best=map5095>best_map5095
        if is_best: best_map5095=map5095
        checkpoint={'epoch':epoch,'model_state':model.state_dict(),'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict(),'scaler_state':scaler.state_dict(),'best_map5095':best_map5095,'history':history,'class_names':CLASS_NAMES,'batch_size':BATCH_SIZE,'grad_accum_steps':GRAD_ACCUM_STEPS,'effective_batch_size':BATCH_SIZE*GRAD_ACCUM_STEPS}
        torch.save(checkpoint,LAST_CKPT)
        if is_best: torch.save(checkpoint,BEST_CKPT); print(f'>>> NEW BEST: mAP50-95={best_map5095:.6f}')
        pd.DataFrame(history).sort_values('epoch').to_csv(HISTORY_CSV,index=False)
        print(f'Epoch {epoch:03d}/{EPOCHS} | loss={train_loss:.4f} | mAP50={map50:.4f} | mAP50-95={map5095:.4f} | time={epoch_minutes:.1f} min')
    print(f'Total training time: {(time.time()-start_all)/60:.1f} min')
else: print('Checkpoint already reached 90 epochs.')

In [ ]:
best=torch.load(BEST_CKPT,map_location=DEVICE,weights_only=False); model.load_state_dict(best['model_state']); model.eval(); final_metrics=validate_map(model,val_loader); print('=== BEST CHECKPOINT / VAL ==='); print('Best epoch:',best['epoch']); print(f"mAP50: {final_metrics['mAP50']:.6f}"); print(f"mAP50-95: {final_metrics['mAP50-95']:.6f}")

In [ ]:
@torch.no_grad()
def calc_precision_recall(model,loader,score_threshold=0.5,iou_threshold=0.5):
    model.eval(); TP=FP=FN=0
    for images,targets in tqdm(loader,desc='VAL P/R',leave=False):
        outputs=model([x.to(DEVICE,non_blocking=True) for x in images])
        for out,tgt in zip(outputs,targets):
            keep=out['scores'].cpu()>=score_threshold; p_boxes=out['boxes'].cpu()[keep]; p_labels=out['labels'].cpu()[keep]; g_boxes=tgt['boxes'].cpu(); g_labels=tgt['labels'].cpu(); matched=set()
            for pb,pl in zip(p_boxes,p_labels):
                best_iou=0.0; best_j=-1
                for j,(gb,gl) in enumerate(zip(g_boxes,g_labels)):
                    if j in matched or int(gl)!=int(pl): continue
                    xa1=max(pb[0],gb[0]); ya1=max(pb[1],gb[1]); xa2=min(pb[2],gb[2]); ya2=min(pb[3],gb[3]); inter=max(0.0,xa2-xa1)*max(0.0,ya2-ya1); area_p=max(0.0,pb[2]-pb[0])*max(0.0,pb[3]-pb[1]); area_g=max(0.0,gb[2]-gb[0])*max(0.0,gb[3]-gb[1]); union=area_p+area_g-inter; iou=inter/union if union>0 else 0.0
                    if iou>best_iou: best_iou=iou; best_j=j
                if best_iou>=iou_threshold and best_j>=0: TP+=1; matched.add(best_j)
                else: FP+=1
            FN += len(g_boxes)-len(matched)
    precision=TP/(TP+FP) if TP+FP else 0.0; recall=TP/(TP+FN) if TP+FN else 0.0; return precision,recall
precision,recall=calc_precision_recall(model,val_loader); print(f'Precision: {precision:.6f}'); print(f'Recall: {recall:.6f}')

In [ ]:
@torch.no_grad()
def measure_inference_ms(model,loader):
    model.eval()
    for i,(images,_) in enumerate(loader):
        imgs=[x.to(DEVICE,non_blocking=True) for x in images]
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        _=model(imgs)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        if i>=2: break
    total_time=0.0; total_images=0
    for images,_ in loader:
        imgs=[x.to(DEVICE,non_blocking=True) for x in images]
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter(); _=model(imgs)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        total_time+=time.perf_counter()-t0; total_images+=len(imgs)
        if total_images>=200: break
    return (total_time/max(total_images,1))*1000.0

inference_ms=measure_inference_ms(model,val_loader); parameters=sum(p.numel() for p in model.parameters())
results=pd.DataFrame([{'model':'Faster R-CNN MobileNetV3-Large FPN','best_epoch':best['epoch'],'epochs':EPOCHS,'parameters':parameters,'parameters_M':parameters/1e6,'physical_batch':BATCH_SIZE,'effective_batch':BATCH_SIZE*GRAD_ACCUM_STEPS,'Precision@0.5':precision,'Recall@0.5':recall,'mAP50':final_metrics['mAP50'],'mAP50-95':final_metrics['mAP50-95'],'mAR100':final_metrics['mAR100'],'inference_ms_per_image':inference_ms}]); results.to_csv(RESULT_CSV,index=False); print(results.T); print('RESULT:',RESULT_CSV); print('BEST:',BEST_CKPT); print('LAST:',LAST_CKPT)

### Ghi chú
Torchvision công bố Faster R-CNN MobileNetV3-Large FPN ở khoảng 19.4M parameters và 4.49 GFLOPS; bản ResNet-50 FPN V2 là 43.7M parameters và 280.37 GFLOPS. Đây là lý do chọn MobileNetV3-Large FPN để làm baseline Faster R-CNN nhẹ hơn cho bài so sánh. citeturn213555search2